In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
 
import torch
from datasets import DatasetDict, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate
import pickle

warnings.filterwarnings("ignore")

In [2]:
train_path = '/Users/philip.juachon/Desktop/Personal Learning/Deep Learning/final-project/output/02_train.jsonl'
val_path = '/Users/philip.juachon/Desktop/Personal Learning/Deep Learning/final-project/output/02_val.jsonl'
test_path = '/Users/philip.juachon/Desktop/Personal Learning/Deep Learning/final-project/output/02_test.jsonl'

In [3]:
train_data = pd.read_json(train_path, lines = True)
val_data = pd.read_json(val_path, lines = True)
test_data = pd.read_json(test_path, lines = True)

In [4]:
train_data.head()

,row_id,headline,category,text_combined
0,153996,Poodle Cat: New Breed Of Feline Is Adorably Fl...,ENVIRONMENT,Poodle Cat: New Breed Of Feline Is Adorably Fl...
1,80071,ESPN Reporter Blasts Cowboys' Sponsors After G...,SPORTS,ESPN Reporter Blasts Cowboys' Sponsors After G...
2,151401,"Palo Verde Beetles, Huge Horny Bugs, Descend O...",ENVIRONMENT,"Palo Verde Beetles, Huge Horny Bugs, Descend O..."
3,177136,The Hope for Brave New Worlds,WELLNESS,The Hope for Brave New Worlds [SEP] The discov...
4,97560,How Many Likes Are Enough?,WELLNESS,How Many Likes Are Enough? [SEP] When we seek ...


In [5]:
print(f"Train Shape: {train_data.shape}")
print(f"Val Shape: {val_data.shape}")
print(f"Test Shape: {test_data.shape}")

Train Shape: (129006, 4)
Val Shape: (32252, 4)
Test Shape: (40315, 4)


In [6]:
with open("/Users/philip.juachon/Desktop/Personal Learning/Deep Learning/final-project/artifacts/02_y_encoder.pkl", "rb") as f:
    y_encoder = pickle.load(f)

In [7]:
train_data['labels'] = train_data['category'].apply(lambda x: y_encoder.transform([x])[0])
val_data['labels'] = val_data['category'].apply(lambda x: y_encoder.transform([x])[0])
test_data['labels'] = test_data['category'].apply(lambda x: y_encoder.transform([x])[0])

In [8]:
train_data.head()

,row_id,headline,category,text_combined,labels
0,153996,Poodle Cat: New Breed Of Feline Is Adorably Fl...,ENVIRONMENT,Poodle Cat: New Breed Of Feline Is Adorably Fl...,7
1,80071,ESPN Reporter Blasts Cowboys' Sponsors After G...,SPORTS,ESPN Reporter Blasts Cowboys' Sponsors After G...,17
2,151401,"Palo Verde Beetles, Huge Horny Bugs, Descend O...",ENVIRONMENT,"Palo Verde Beetles, Huge Horny Bugs, Descend O...",7
3,177136,The Hope for Brave New Worlds,WELLNESS,The Hope for Brave New Worlds [SEP] The discov...,22
4,97560,How Many Likes Are Enough?,WELLNESS,How Many Likes Are Enough? [SEP] When we seek ...,22


In [9]:
data = DatasetDict({'train': Dataset.from_pandas(train_data[["row_id",'text_combined','labels']]),
                    'validation': Dataset.from_pandas(val_data[["row_id",'text_combined','labels']]),
                    'test': Dataset.from_pandas(test_data[["row_id",'text_combined','labels']])})

data

DatasetDict({
    train: Dataset({
        features: ['row_id', 'text_combined', 'labels'],
        num_rows: 129006
    })
    validation: Dataset({
        features: ['row_id', 'text_combined', 'labels'],
        num_rows: 32252
    })
    test: Dataset({
        features: ['row_id', 'text_combined', 'labels'],
        num_rows: 40315
    })
})

In [10]:
data['train']['labels'][:5]

[7, 17, 7, 22, 22]

## Tokenize

In [11]:
MODEL_NAME = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [12]:
def preprocess_data(examples):
    # take a batch of texts
    text = examples['text_combined']
    # encode them
    encoding = tokenizer(text, padding="max_length", truncation=True, max_length=256)
    
    # add labels
    encoding['labels'] = torch.Tensor(examples['labels'])

    return encoding

In [13]:
encoded_dataset = data.map(preprocess_data,
                           batched=True, 
                           remove_columns = data['train'].column_names)

Map:   0%|          | 0/129006 [00:00<?, ? examples/s]

Map:   0%|          | 0/32252 [00:00<?, ? examples/s]

Map:   0%|          | 0/40315 [00:00<?, ? examples/s]

In [14]:
import os
os.getcwd()

'/Users/philip.juachon/Desktop/Personal Learning/Deep Learning/final-project'

In [15]:
encoded_dataset.save_to_disk("title_classifier_encoded_dataset_distilroberta.hf")

Saving the dataset (0/1 shards):   0%|          | 0/129006 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/32252 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/40315 [00:00<?, ? examples/s]